# Prism CUDA Validation
**Runtime → Change runtime type → A100 or T4**

Prism runs FIRST (the interesting result). Ortho baseline runs second (can run overnight).
Each run saves to `/content/` immediately — survives kernel restart.

In [ ]:
# Cell 1: Setup
import os, subprocess
if os.path.exists('/content/prism'):
    subprocess.run(['rm', '-rf', '/content/prism'])
!git clone https://github.com/realityinspector/prismic-pretraining.git /content/prism
%cd /content/prism
!pip install -q transformers datasets scipy
import torch
gpu = torch.cuda.get_device_name(0)
print(f'GPU: {gpu}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')
from transformers import GPT2TokenizerFast, GPT2LMHeadModel
GPT2TokenizerFast.from_pretrained('gpt2')
GPT2LMHeadModel.from_pretrained('gpt2')
print('Ready.')

In [ ]:
# Cell 2: Run PRISM first (the interesting result)
import sys, os, json, gc, time, warnings
import torch
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.pretrained_extract import extract_per_layer, make_hybrid_init_fn

install_signal_handlers()
device = 'cuda'

STEPS = 2000
EVALS = [250, 500, 750, 1000, 1500, 2000]

base_config = dict(
    max_steps=STEPS,
    eval_steps=EVALS,
    warmup_steps=200,
    log_every=50,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

print('='*60)
print('  PRISM: Spectral Imprint + EigenTransfer + 2x LR')
print('='*60)

with open('prism/results/extracted_spectra.json') as f:
    extracted = json.load(f)
print('Extracting pretrained directions for EigenTransfer...')
dirs = extract_per_layer('gpt2', include_directions=True, device='cpu')
init_fn = make_hybrid_init_fn(
    extracted['spectra_coeffs'], dirs,
    lam=1.0, align_mode='UV', align_strength=0.5
)
gc.collect()

t0 = time.time()
result_prism = train(TrainConfig(**base_config, lr=1.25e-4),
                     init_fn=init_fn,
                     init_name='prism', verbose=True)
elapsed = time.time() - t0
print(f'\nPrism done in {elapsed:.0f}s — Final PPL: {result_prism["final_ppl"]:.1f}')

prism_data = {
    'name': 'prism',
    'final_ppl': result_prism['final_ppl'],
    'elapsed': elapsed,
    'checkpoints': {str(k): v for k, v in result_prism['checkpoints'].items()},
    'gpu': torch.cuda.get_device_name(0),
    'steps': STEPS, 'batch': 64, 'seq_len': 1024, 'lr': 1.25e-4,
}
with open('/content/prism_result.json', 'w') as f:
    json.dump(prism_data, f, indent=2)
print('SAVED: /content/prism_result.json')
_clear_memory(device); gc.collect()

In [ ]:
# Cell 3: Run ORTHO baseline (leave running overnight)
import sys, os, json, gc, time, warnings
import torch, numpy as np
warnings.filterwarnings('ignore')
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

sys.path.insert(0, '/content/prism')
os.chdir('/content/prism')

from prism.config import TrainConfig, install_signal_handlers
from prism.train import train, _clear_memory
from prism.baselines import make_init_fn

install_signal_handlers()
device = 'cuda'

STEPS = 2000
EVALS = [250, 500, 750, 1000, 1500, 2000]

base_config = dict(
    max_steps=STEPS,
    eval_steps=EVALS,
    warmup_steps=200,
    log_every=50,
    seed=42,
    device=device,
    batch_size=4,
    grad_accum_steps=16,
    max_length=1024,
    memory_pressure_threshold=5,
)

print('='*60)
print('  ORTHO: Orthogonal baseline (lr=6.25e-5)')
print('='*60)
t0 = time.time()
result_ortho = train(TrainConfig(**base_config, lr=6.25e-5),
                     init_fn=make_init_fn('orthogonal'),
                     init_name='ortho', verbose=True)
elapsed = time.time() - t0
print(f'\nOrtho done in {elapsed:.0f}s — Final PPL: {result_ortho["final_ppl"]:.1f}')

ortho_data = {
    'name': 'ortho',
    'final_ppl': result_ortho['final_ppl'],
    'elapsed': elapsed,
    'checkpoints': {str(k): v for k, v in result_ortho['checkpoints'].items()},
    'gpu': torch.cuda.get_device_name(0),
    'steps': STEPS, 'batch': 64, 'seq_len': 1024, 'lr': 6.25e-5,
}
with open('/content/ortho_result.json', 'w') as f:
    json.dump(ortho_data, f, indent=2)
print('SAVED: /content/ortho_result.json')
_clear_memory(device); gc.collect()

In [ ]:
# Cell 4: Compare (reads from disk)
import json
prism = json.load(open('/content/prism_result.json'))
ortho = json.load(open('/content/ortho_result.json'))
print('='*50)
print(f'GPU: {prism["gpu"]}')
print(f'\n{"Step":>6s}  {"Ortho":>8s}  {"Prism":>8s}  {"Ratio":>7s}')
print('-' * 35)
for s in ['250','500','750','1000','1500','2000']:
    o = ortho['checkpoints'].get(s)
    p = prism['checkpoints'].get(s)
    if o and p:
        print(f'{s:>6s}  {o:>8.1f}  {p:>8.1f}  {o/p:>6.2f}x')
r = ortho['final_ppl'] / prism['final_ppl']
print(f'\nFinal: ortho={ortho["final_ppl"]:.1f}  prism={prism["final_ppl"]:.1f}  ratio={r:.2f}x')

In [ ]:
# Cell 5: Download
from google.colab import files
for f in ['/content/prism_result.json', '/content/ortho_result.json']:
    try: files.download(f)
    except: print(f'{f} not found')